In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.transforms import functional as F
from PIL import Image
import copy

In [ ]:
CROP_SIZE_LR = 96
CROP_SIZE_HR = CROP_SIZE_LR * 2
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN_RGB = [0.4488, 0.4371, 0.4040]
CHECKPOINT_DIR = r"/content/drive/MyDrive/Datasets/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, num_features):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu(out)
        out = self.conv2(out)
        out = out * 0.1 
        return out + residual
    
class EDSR(nn.Module):
    def __init__(self, num_blocks=32, num_features=256, scale_factor=2):
        super(EDSR, self).__init__()
        self.input_conv = nn.Conv2d(3, num_features, kernel_size=3, padding=1)

        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(num_features) for _ in range(num_blocks)]
        )

        self.output_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.upsample = nn.Conv2d(num_features, 3 * (scale_factor ** 2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)

    def forward(self, x):
        x = self.input_conv(x)
        residual = x
        x = self.residual_blocks(x)
        x = self.output_conv(x)
        x += residual
        x = self.upsample(x)
        x = self.pixel_shuffle(x)
        return x

In [ ]:
def data_generator(hr_dir, lr_dir, batch_size):
    hr_files = sorted([f for f in os.listdir(hr_dir) if f.endswith('.png')])
    lr_files = sorted([f for f in os.listdir(lr_dir) if f.endswith('.png')])

    while True:
        hr_batch, lr_batch = [], []

        for _ in range(batch_size):
            idx = random.randint(0, len(hr_files) - 1)
            hr_img = Image.open(os.path.join(hr_dir, hr_files[idx])).convert('RGB')
            lr_img = Image.open(os.path.join(lr_dir, lr_files[idx])).convert('RGB')

            hr_w, hr_h = hr_img.size
            lr_w, lr_h = hr_w // 2, hr_h // 2

            x = random.randint(0, lr_w - CROP_SIZE_LR)
            y = random.randint(0, lr_h - CROP_SIZE_LR)

            hr_crop = hr_img.crop((x * 2, y * 2, x * 2 + CROP_SIZE_HR, y * 2 + CROP_SIZE_HR))
            lr_crop = lr_img.crop((x, y, x + CROP_SIZE_LR, y + CROP_SIZE_LR))

            #data augmentation
            if random.random() > 0.5:
                hr_crop = F.hflip(hr_crop)
                lr_crop = F.hflip(lr_crop)
            if random.random() > 0.5:
                hr_crop = F.vflip(hr_crop)
                lr_crop = F.vflip(lr_crop)
            if random.random() > 0.5:
                hr_crop = hr_crop.rotate(90)
                lr_crop = lr_crop.rotate(90)

            hr_crop = F.to_tensor(hr_crop)
            lr_crop = F.to_tensor(lr_crop)
            hr_crop = hr_crop - torch.tensor(MEAN_RGB).view(3, 1, 1)
            lr_crop = lr_crop - torch.tensor(MEAN_RGB).view(3, 1, 1)

            hr_batch.append(hr_crop)
            lr_batch.append(lr_crop)

        yield torch.stack(lr_batch), torch.stack(hr_batch)


In [ ]:
def train_and_validate(model, train_gen, val_gen, num_epochs=100, val_steps=10):
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999), eps=1e-8)

    best_model_wts = copy.deepcopy(model.state_dict())
    lowest_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for step in range(50):
            lr, hr = next(train_gen)
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)

            # Forward pass
            output = model(lr)
            loss = criterion(output, hr)
            train_loss += loss.item()

            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_loss /= 50 
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for _ in range(val_steps):
                lr, hr = next(val_gen)
                lr, hr = lr.to(DEVICE), hr.to(DEVICE)

                output = model(lr)
                loss = criterion(output, hr)
                val_loss += loss.item()

        val_loss /= val_steps  
        print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {val_loss:.4f}")
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'edsr_x2.pth'))
        if val_loss < lowest_val_loss:
            lowest_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'edsr_x2_best.pth'))
            print("Model checkpoint saved.")

    # Load best model weights
    model.load_state_dict(best_model_wts)
    print("Training complete. Best validation loss:", lowest_val_loss)
    return model


In [ ]:
model = EDSR().to(DEVICE)

train_gen = data_generator(r"/content/drive/MyDrive/Datasets/DIV2K/Train/DIV2K_train_HR", r"/content/drive/MyDrive/Datasets/DIV2K/Train/DIV2K_train_LR_bicubic/X2", BATCH_SIZE)
val_gen = data_generator(r"/content/drive/MyDrive/Datasets/DIV2K/Test/DIV2K_valid_HR", r"/content/drive/MyDrive/Datasets/DIV2K/Test/DIV2K_valid_LR_bicubic/X2", BATCH_SIZE)  # Use separate val set in practice

trained_model = train_and_validate(model, train_gen, val_gen)

In [ ]:
#peace